In [1]:
# Notebook setup: import and (optionally) auto-reload local module
import importlib
import time

import locator_time
import locator_time2

# If you edit locator_time2.py while this notebook is open, re-run this cell to reload.
importlib.reload(locator_time2)

print(locator_time2.__doc__[:400])

A small, self-contained Python port of ImPlot's time-axis tick locator.

This is based on the logic in `implot.cpp` (function `Locator_Time`) and the
associated helpers in the "Time Ticks and Utils" section.

Goal
----
Given:
  - `t_min` and `t_max` as Unix timestamps in seconds (float or int)
  - `pixels` as the axis pixel length (float or int)

Produce:
  - tick positions (float seconds)
  - lab


## Test out the Data Structures

In [2]:
# test out ImPlotTime from locator_time2
locator_time2.ImPlotTime(1, 1)

ImPlotTime(S=1, Us=1)

In [3]:
# arithmetic + normalization
(locator_time2.ImPlotTime(3, 1) + locator_time2.ImPlotTime(1, 2_000_000)).to_double()

6.000001

## Test out the helper functions

In [4]:
# Small helpers to inspect returned ticks
from collections import Counter

def summarize_ticks(ticks):
    c = Counter((t.level, t.major, t.show_label) for t in ticks)
    total = len(ticks)
    level0 = sum(1 for t in ticks if t.level == 0)
    level1 = sum(1 for t in ticks if t.level == 1)
    shown = sum(1 for t in ticks if t.show_label)
    return {
        'total_ticks': total,
        'level0_ticks': level0,
        'level1_ticks': level1,
        'labels_shown': shown,
        'breakdown': dict(c),
    }

def head_ticks(ticks, n=20):
    rows = []
    for t in ticks[:n]:
        tag = f"L{t.level} {'M' if t.major else 'm'}"
        label = t.label if t.show_label else ''
        rows.append((tag, t.pos, label))
    return rows

## Baseline: 1 hour range
This should produce minute/second-ish ticks depending on `pixels` and `max_density`.

In [5]:
now = time.time()
t_min = now
t_max = now + 3600

ticks = locator_time2.locator_time(
    t_min, t_max, pixels=800,
    use_local_time=True,
    max_density=0.5,
    char_px=7.0,
)

summarize_ticks(ticks), head_ticks(ticks, 25)

({'total_ticks': 8,
  'level0_ticks': 6,
  'level1_ticks': 2,
  'labels_shown': 7,
  'breakdown': {(0, False, True): 5,
   (1, True, True): 1,
   (0, True, True): 1,
   (1, True, False): 1}},
 [('L0 m', 1767115800.0, '12:30pm'),
  ('L1 M', 1767115800.0, '12/30/25'),
  ('L0 m', 1767116400.0, '12:40pm'),
  ('L0 m', 1767117000.0, '12:50pm'),
  ('L0 M', 1767117600.0, '1:00pm'),
  ('L1 M', 1767117600.0, ''),
  ('L0 m', 1767118200.0, '1:10pm'),
  ('L0 m', 1767118800.0, '1:20pm')])

## Compare pixel widths
Smaller `pixels` should suppress more labels (especially level 0 minor labels).

In [6]:
for px in (200, 400, 800, 1200):
    ticks_px = locator_time2.locator_time(t_min, t_max, pixels=px, use_local_time=True)
    s = summarize_ticks(ticks_px)
    print(f"pixels={px:4d}  total={s['total_ticks']:4d}  shown={s['labels_shown']:4d}  L0={s['level0_ticks']:4d}  L1={s['level1_ticks']:4d}")

pixels= 200  total=   4  shown=   3  L0=   2  L1=   2
pixels= 400  total=   6  shown=   5  L0=   4  L1=   2
pixels= 800  total=   8  shown=   7  L0=   6  L1=   2
pixels=1200  total=  14  shown=  13  L0=  12  L1=   2


## Time the locator_time function

In [7]:
import timeit

In [8]:
# use timeit magic to benchmark locator_time2.locator_time
timeit.timeit(
    "locator_time2.locator_time(t_min, t_max, pixels=800, use_local_time=True)",
    globals=globals(),
    number=1000,
)

0.12287280010059476

## Explore different spans
These cover typical unit transitions (minutes → hours → days → months → years).

In [9]:
def run_span(span_seconds, pixels=900, title=None):
    t0 = time.time()
    t1 = t0 + span_seconds
    ticks = locator_time2.locator_time(t0, t1, pixels=pixels, use_local_time=True)
    s = summarize_ticks(ticks)
    title = title or f"span={span_seconds}s"
    print(f"\n{title} (pixels={pixels})")
    print(f"  total={s['total_ticks']}  shown={s['labels_shown']}")
    print('  first 12:', head_ticks(ticks, 12))

run_span(10, title='10 seconds')
run_span(5 * 60, title='5 minutes')
run_span(6 * 3600, title='6 hours')
run_span(2 * 86400, title='2 days')
run_span(45 * 86400, title='45 days')
run_span(400 * 86400, title='~400 days (year-ish)')
run_span(10 * 365 * 86400, title='~10 years (year locator)')


10 seconds (pixels=900)
  total=11  shown=11
  first 12: [('L0 m', 1767116109.0, ':09'), ('L1 M', 1767116109.0, '12/30/25 12:35pm'), ('L0 m', 1767116110.0, ':10'), ('L0 m', 1767116111.0, ':11'), ('L0 m', 1767116112.0, ':12'), ('L0 m', 1767116113.0, ':13'), ('L0 m', 1767116114.0, ':14'), ('L0 m', 1767116115.0, ':15'), ('L0 m', 1767116116.0, ':16'), ('L0 m', 1767116117.0, ':17'), ('L0 m', 1767116118.0, ':18')]

5 minutes (pixels=900)
  total=26  shown=26
  first 12: [('L0 m', 1767116115.0, ':15'), ('L1 M', 1767116115.0, '12/30/25 12:35pm'), ('L0 m', 1767116130.0, ':30'), ('L0 m', 1767116145.0, ':45'), ('L0 M', 1767116160.0, ':00'), ('L1 M', 1767116160.0, '12:36pm'), ('L0 m', 1767116175.0, ':15'), ('L0 m', 1767116190.0, ':30'), ('L0 m', 1767116205.0, ':45'), ('L0 M', 1767116220.0, ':00'), ('L1 M', 1767116220.0, '12:37pm'), ('L0 m', 1767116235.0, ':15')]

6 hours (pixels=900)
  total=12  shown=7
  first 12: [('L0 M', 1767117600.0, '1:00pm'), ('L1 M', 1767117600.0, '12/30/25'), ('L0 M', 17

## ISO-8601 / 24-hour formatting toggles
These flags match the knobs you might want in a UI layer.

In [10]:
t_min = time.time()
t_max = t_min + 3 * 3600

ticks_default = locator_time2.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=False, use_iso8601=False)
ticks_iso24 = locator_time2.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=True, use_iso8601=True)

print('default:', head_ticks(ticks_default, 10))
print('iso+24:', head_ticks(ticks_iso24, 10))

default: [('L0 M', 1767117600.0, '1:00pm'), ('L1 M', 1767117600.0, '12/30/25'), ('L0 m', 1767119400.0, '1:30pm'), ('L0 M', 1767121200.0, '2:00pm'), ('L1 M', 1767121200.0, ''), ('L0 m', 1767123000.0, '2:30pm'), ('L0 M', 1767124800.0, '3:00pm'), ('L1 M', 1767124800.0, ''), ('L0 m', 1767126600.0, '3:30pm')]
iso+24: [('L0 M', 1767117600.0, '13:00'), ('L1 M', 1767117600.0, '2025-12-30'), ('L0 m', 1767119400.0, '13:30'), ('L0 M', 1767121200.0, '14:00'), ('L1 M', 1767121200.0, ''), ('L0 m', 1767123000.0, '14:30'), ('L0 M', 1767124800.0, '15:00'), ('L1 M', 1767124800.0, ''), ('L0 m', 1767126600.0, '15:30')]


## TimeAxisLocator wrapper
This exercises the reusable class intended for fast zoom callbacks.

In [11]:
loc = locator_time2.TimeAxisLocator(use_local_time=True, prewarm=True)
ticks2 = loc(time.time(), time.time() + 3600, 800)
summarize_ticks(ticks2), head_ticks(ticks2, 25)

({'total_ticks': 8,
  'level0_ticks': 6,
  'level1_ticks': 2,
  'labels_shown': 7,
  'breakdown': {(0, False, True): 5,
   (1, True, True): 1,
   (0, True, True): 1,
   (1, True, False): 1}},
 [('L0 m', 1767116400.0, '12:40pm'),
  ('L1 M', 1767116400.0, '12/30/25'),
  ('L0 m', 1767117000.0, '12:50pm'),
  ('L0 M', 1767117600.0, '1:00pm'),
  ('L1 M', 1767117600.0, ''),
  ('L0 m', 1767118200.0, '1:10pm'),
  ('L0 m', 1767118800.0, '1:20pm'),
  ('L0 m', 1767119400.0, '1:30pm')])

## Optional: PIL-based text measurement
If you want more ImPlot-like behavior, measure string widths using the same font file + size your DearCyGui axis labels use.

This requires Pillow (`pip install pillow`). If Pillow/font loading fails, it will fall back to the `char_px` estimator.

In [12]:
# TODO: set these to match your DearCyGui axis font
font_path = r"C:\Users\khazy\OneDrive\Documents\DCG_Release_Tests_py11\.venv\Lib\site-packages\dearcygui\lmsans17-regular.otf"  # e.g. r"C:\\path\\to\\yourfont.otf"
#font_path = None
font_size_px = 17  # e.g. 17

measure = None
if font_path is not None and font_size_px is not None:
    try:
        measure = locator_time2.make_pil_text_width_measurer(font_path, font_size_px)
        print('PIL measurer enabled')
    except Exception as e:
        print('PIL measurer not available, falling back:', e)
        measure = None

loc_pil = locator_time2.TimeAxisLocator(
    use_local_time=True,
    measure_text_width_px=measure,
    prewarm=True,
)
ticks_pil = loc_pil(time.time(), time.time() + 3600, 800)
summarize_ticks(ticks_pil), head_ticks(ticks_pil, 25)

PIL measurer enabled


({'total_ticks': 8,
  'level0_ticks': 6,
  'level1_ticks': 2,
  'labels_shown': 7,
  'breakdown': {(0, False, True): 5,
   (1, True, True): 1,
   (0, True, True): 1,
   (1, True, False): 1}},
 [('L0 m', 1767116400.0, '12:40pm'),
  ('L1 M', 1767116400.0, '12/30/25'),
  ('L0 m', 1767117000.0, '12:50pm'),
  ('L0 M', 1767117600.0, '1:00pm'),
  ('L1 M', 1767117600.0, ''),
  ('L0 m', 1767118200.0, '1:10pm'),
  ('L0 m', 1767118800.0, '1:20pm'),
  ('L0 m', 1767119400.0, '1:30pm')])

In [20]:
type(measure)

functools._lru_cache_wrapper

In [26]:
measure("Hi there")

53.0

### testing text width functions

In [32]:
from locator_time2 import make_pil_text_width_measurer, make_time, TIME_FORMAT_LEVEL0, TIME_FORMAT_LEVEL1, TIME_FORMAT_LEVEL1_FIRST, format_datetime
from PIL import ImageFont

#### ImageFont bounding box

In [19]:
font = ImageFont.truetype(font_path, font_size_px)

text = "12:58:58.888"
bbox = font.getbbox(text)
bbox

(0, 8, 84, 20)

In [20]:
(bbox[2] - bbox[0])

84

#### Measurer from locator_time2

In [26]:
measurer = make_pil_text_width_measurer(font_path, font_size_px)

In [27]:
texts = ["12:58:58.888", "Jan 2025", "2025-12-30 23:59:59"]
widths = [measurer(t) for t in texts]

In [28]:
widths

[84.0, 61.0, 135.0]

In [29]:
list(zip(texts, widths))

[('12:58:58.888', 84.0), ('Jan 2025', 61.0), ('2025-12-30 23:59:59', 135.0)]

In [30]:
measurer.cache_info()     # hits/misses/maxsize/currsize

CacheInfo(hits=0, misses=3, maxsize=2048, currsize=3)

In [33]:
t = make_time(2888, 11, 22, 12, 58, 58, 888888, use_local_time=False)

specs = [*TIME_FORMAT_LEVEL0, *TIME_FORMAT_LEVEL1, *TIME_FORMAT_LEVEL1_FIRST]
labels = [format_datetime(t, spec, use_local_time=False, use_24_hour=True, use_iso8601=False) for spec in specs]
widths = [measurer(lbl) for lbl in labels]

widths

[57.0,
 48.0,
 20.0,
 36.0,
 36.0,
 40.0,
 26.0,
 32.0,
 36.0,
 56.0,
 36.0,
 36.0,
 64.0,
 64.0,
 32.0,
 32.0,
 125.0,
 125.0,
 105.0,
 105.0,
 64.0,
 64.0,
 32.0,
 32.0]

In [37]:
labels

['.888 888',
 ':58.888',
 ':58',
 '12:58',
 '12:00',
 '12/22',
 'Dec',
 '2888',
 '12:58',
 '12:58:58',
 '12:58',
 '12:58',
 '12/22/88',
 '12/22/88',
 '2888',
 '2888',
 '12/22/88 12:58:58',
 '12/22/88 12:58:58',
 '12/22/88 12:58',
 '12/22/88 12:58',
 '12/22/88',
 '12/22/88',
 '2888',
 '2888']

In [38]:
t

ImPlotTime(S=29000120338, Us=888888)